<a href="https://colab.research.google.com/github/Rahul9994/ML_Flyrank/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import os
REPO_URL = "https://github.com/Rahul9994/ML_Flyrank"
REPO_DIR = "ML_Flyrank"
if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)

!pip install duckdb --quiet
import duckdb
from google.colab import userdata
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
base = "hf://datasets/FlyRank/internship-warehouse"

Cloning into 'ML_Flyrank'...
remote: Enumerating objects: 90, done.
remote: Counting objects: 100% (90/90), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 90 (delta 11), reused 73 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (90/90), 1.85 MiB | 11.87 MiB/s, done.
Resolving deltas: 100% (11/11), done.


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Methodology question: Since health_score is arithmetically built from position,
impressions, ctr, and scroll depth, how much of the reported 43%+32% importance on
position and impressions reflects genuine predictive signal about content quality,
versus the model simply re-discovering the label's own construction formula? This is
the same feature-vs-label leakage pattern I checked for in my own Week-3/Week-6 audit.

Finding A: Random Forest feature importance for health score (ML Appendix). The paper itself flags this honestly: the model is holdout-tested, but the target itself is partly constructed from some of these inputs, so importance is descriptive rather than causal. Since Average Position is the #1 predictor of health score at 43% importance, followed by Impressions (32%), and Health Score is explicitly defined as Impressions (30 pts) + position (30 pts) + CTR (20 pts) + scroll depth (20 pts), two of the "predictors" are literally components of the label's own formula.

Finding B: Logistic regression growth classifier (71% holdout accuracy). The paper reports Content Age, Days Since Update, and Days Visible as the strongest signals separating growing from declining pages, using an 80/20 split from 61.8K content pieces across only 57 brands.

Methodology question: With only 57 brands contributing 61.8K rows, was the 80/20
split grouped by brand, or a plain random row split? If random, the same brand's
near-duplicate pages could appear in both train and test, letting the model partly
learn brand-level baseline behavior rather than a generalizable growth pattern — the
same grouped-split concern I addressed in my own Week-5/Week-6 model validation.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Comparing a naive random split against a grouped (client-aware) split reveals a
meaningful gap: R2 = 0.0360 under the naive split versus R2 = 0.0076 under the
grouped split — roughly a 4.7x drop. This confirms that a plain random split
overstates model performance here, since the same client can appear in both train
and test, letting the model partly learn client-specific patterns rather than
signal that generalizes to entirely new clients. The grouped split is the honest
number, and it shows my model's real predictive power over the baseline is
smaller than the naive evaluation would suggest — a modest but real signal, not
a strong one.

In [3]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error

df = con.sql(f"""
    SELECT client_hash_id, content_hash_id, avg_position_90d,
           content_total_impressions_90d, query_char_count, query_token_count,
           rare_impressions_share, anonymized_impressions_share,
           clicks_90d * 1.0 / NULLIF(impressions_90d,0) AS ctr
    FROM read_parquet('{base}/fact_content_query_90d.parquet')
    WHERE impressions_90d > 0 AND avg_position_90d >= 1
    USING SAMPLE 300000 ROWS
""").df()

features = ["avg_position_90d", "content_total_impressions_90d", "query_char_count",
            "query_token_count", "rare_impressions_share", "anonymized_impressions_share"]

# BEFORE: naive random split
Xtr, Xte, ytr, yte = train_test_split(df[features].fillna(0), df["ctr"], test_size=0.2, random_state=42)
model_naive = GradientBoostingRegressor(random_state=42).fit(Xtr, ytr)
r2_naive = r2_score(yte, model_naive.predict(Xte))

# AFTER: grouped split (your Week-5 approach)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, te_idx = next(gss.split(df, groups=df["client_hash_id"]))
Xtr2, Xte2 = df[features].fillna(0).iloc[tr_idx], df[features].fillna(0).iloc[te_idx]
ytr2, yte2 = df["ctr"].iloc[tr_idx], df["ctr"].iloc[te_idx]
model_grouped = GradientBoostingRegressor(random_state=42).fit(Xtr2, ytr2)
r2_grouped = r2_score(yte2, model_grouped.predict(Xte2))

print(f"Naive random split R2: {r2_naive:.4f}")
print(f"Grouped (honest) split R2: {r2_grouped:.4f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Naive random split R2: 0.0360
Grouped (honest) split R2: 0.0076


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

All six features show weak correlations with ctr (ranging -0.12 to +0.05),
consistent with genuine predictive signal rather than label leakage — a
leaked feature would show a correlation near 1.0. No feature is derived from
clicks_90d or impressions_90d after the fact.

In [4]:
for col in features:
    print(f"{col}: correlation with ctr = {df[col].corr(df['ctr']):.4f}")
print("\nNone of these features are computed from clicks_90d or impressions_90d "
      "post-hoc — same conclusion as Week-3's audit, reapplied to the final feature set.")

avg_position_90d: correlation with ctr = -0.1172
content_total_impressions_90d: correlation with ctr = 0.0311
query_char_count: correlation with ctr = -0.0193
query_token_count: correlation with ctr = -0.0135
rare_impressions_share: correlation with ctr = -0.0551
anonymized_impressions_share: correlation with ctr = 0.0516

None of these features are computed from clicks_90d or impressions_90d post-hoc — same conclusion as Week-3's audit, reapplied to the final feature set.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original (Week-5): "This confirms the model captures some real signal beyond a flat average."
Rewritten: Under an honest, client-grouped validation split, the model's improvement
over the baseline is small (R2 = 0.0076) — real but modest, and considerably weaker
than a naive random split would suggest (R2 = 0.0360). This should be read as weak
decision-support evidence for prioritization, not proof of strong predictive power,
consistent with the paper's own caution that its ML appendix results are exploratory
and descriptive rather than causal.

## Self-check

Before you submit, confirm each line honestly:
- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.